**Loading the Preprocessed Data**

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load your preprocessed dataset
df = pd.read_csv('political_wiki_human_ai_dataset.csv')


**Splitting the 20% of the data for Testing**

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['label'],
    test_size=0.2,  # 20% for testing
    random_state=42,
    stratify=df['label']  # keeps class balance
)


**TF-IDF vectorization**

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create TF-IDF vectorizer
vectorizer = TfidfVectorizer(
    ngram_range=(1,2),  # unigrams and bigrams
    min_df=3,           # only include words in at least 3 docs
    max_df=0.9,         # exclude very common words
    max_features=10000  # adjust if memory is an issue
)

# Fit on training, transform both train and test
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)


In [7]:
print(vectorizer.vocabulary_["nato"])

5508


**Classify using Logistic Regression**

In [9]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train_vec, y_train)


LogisticRegression(max_iter=1000, random_state=42)

In [13]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

y_pred = clf.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=['Human', 'AI']))


Accuracy: 0.9965034965034965

Confusion Matrix:
 [[284   2]
 [  0 286]]

Classification Report:
               precision    recall  f1-score   support

       Human       1.00      0.99      1.00       286
          AI       0.99      1.00      1.00       286

    accuracy                           1.00       572
   macro avg       1.00      1.00      1.00       572
weighted avg       1.00      1.00      1.00       572



In [15]:
import joblib
joblib.dump(clf, 'tfidf_logreg_model.joblib')
joblib.dump(vectorizer, 'tfidf_vectorizer.joblib')


['tfidf_vectorizer.joblib']

In [17]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=['Human', 'AI']))


Accuracy: 0.9965034965034965

Classification Report:
               precision    recall  f1-score   support

       Human       1.00      0.99      1.00       286
          AI       0.99      1.00      1.00       286

    accuracy                           1.00       572
   macro avg       1.00      1.00      1.00       572
weighted avg       1.00      1.00      1.00       572



In [19]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(clf, vectorizer.transform(df['text']), df['label'], cv=5)
print("Cross-validation accuracy:", scores.mean())


Cross-validation accuracy: 0.9951048951048952


**Identifying the Misclassified Texts**

In [21]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load your dataset
df = pd.read_csv('political_wiki_human_ai_dataset.csv')

# Split the data exactly as you did for training/testing
X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['label'], test_size=0.2, random_state=42, stratify=df['label']
)

# You need the test set's indices to get the titles!
test_indices = y_test.index

# Predict using your model (clf, vectorizer as before)
X_test_vec = vectorizer.transform(X_test)
y_pred = clf.predict(X_test_vec)

# Build a DataFrame for easy comparison
test_results = pd.DataFrame({
    'title': df.loc[test_indices, 'title'].values,
    'true_label': y_test.values,
    'pred_label': y_pred,
    'text': X_test.values
})

# Show misclassified samples
misclassified = test_results[test_results['true_label'] != test_results['pred_label']]

print("Misclassified samples (title and true/predicted labels):")
print(misclassified[['title', 'true_label', 'pred_label']])


Misclassified samples (title and true/predicted labels):
                          title  true_label  pred_label
38    Royal Jordanian Air Force           0           1
108  Foreign relations of Samoa           0           1


**Installing the Pretrained model -> sentence-transformers**

In [1]:
pip install sentence-transformers scikit-learn



   ---------------------------------------- 0.0/470.2 kB ? eta -:--:--
    --------------------------------------- 10.2/470.2 kB ? eta -:--:--
    --------------------------------------- 10.2/470.2 kB ? eta -:--:--
   ----- --------------------------------- 71.7/470.2 kB 653.6 kB/s eta 0:00:01
   ------------------------------- -------- 368.6/470.2 kB 2.3 MB/s eta 0:00:01
   ---------------------------------------- 470.2/470.2 kB 2.5 MB/s eta 0:00:00


**Classification using sentence-transformers**

In [47]:
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import os

# Load article texts
def load_articles(folder_path, label):
    texts = []
    labels = []
    for filename in os.listdir(folder_path):
        if filename.endswith(".txt"):
            with open(os.path.join(folder_path, filename), "r", encoding="utf-8") as f:
                texts.append(f.read())
                labels.append(label)
    return texts, labels

human_texts, human_labels = load_articles("./Dataset_worldPolitics/wiki_old_politics_dataset", 0)
ai_texts, ai_labels = load_articles("./Dataset_worldPolitics/ai_politics_dataset", 1)

# Combine
texts = human_texts + ai_texts
labels = human_labels + ai_labels

# Get sentence embeddings
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(texts, show_progress_bar=True)

# Split data
X_train, X_test, y_train, y_test = train_test_split(embeddings, labels, test_size=0.2, random_state=42)

# Train Logistic Regression
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

# Evaluate
y_pred = clf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))


Batches:   0%|          | 0/90 [00:00<?, ?it/s]

Accuracy: 0.8846153846153846
Classification Report:
               precision    recall  f1-score   support

           0       0.90      0.87      0.89       295
           1       0.87      0.90      0.88       277

    accuracy                           0.88       572
   macro avg       0.88      0.89      0.88       572
weighted avg       0.89      0.88      0.88       572

